In [ ]:
!pip install -q --upgrade transformers==4.45.0 huggingface_hub accelerate

# SemEval-2026 Task 13 — Subtask C: Improved Hybrid Code Detection

**Goal:** 4-class classification — Human / Machine-Generated / Hybrid / Adversarial

### Improvements over baseline (CodeBERT + linear head → macro-F1 0.71)
| # | Technique | Purpose |
|---|-----------|--------|
| 1 | **Backbone swap** — UniXcoder / GraphCodeBERT | Richer code semantics via DFG/AST-aware pretraining |
| 2 | **Bi-LSTM classification head** | Capture sequential patterns across full hidden sequence |
| 3 | **Multi-Dense-Block head** | Deeper, residual decision boundary on [CLS] |
| 4 | **Focal Loss + class weights** | Down-weight easy examples, up-weight hybrid/adversarial |
| 5 | **Label smoothing** (configurable) | Prevent overconfidence on majority class |
| 6 | **Layer-wise LR Decay (LLRD)** | Fine-grained adaptation per transformer layer |
| 7 | **Supervised Contrastive Loss** | Pull same-class embeddings together, push different apart |
| 8 | **Per-class threshold calibration** | Maximise macro-F1 post-training |
| 9 | **10% data sample + max_length 256 (fast mode)** | Fast iteration for prototyping; scale to 100% for final run |

In [ ]:
import os
os.environ["WANDB_DISABLED"] = "true"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"
os.environ["XLA_FLAGS"] = "--xla_gpu_cuda_data_dir=''"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import logging
logging.disable(logging.WARNING)

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoConfig,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    DataCollatorWithPadding,
)
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
)
from sklearn.utils.class_weight import compute_class_weight
import warnings
warnings.filterwarnings("ignore")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Configuration

In [ ]:
# ─── Data paths ──────────────────────────────────────────────────────

# --- OPTION A: Kaggle input paths ---
# TRAIN_PATH = "/kaggle/input/datasets/vortex07/kaggle/task_c_training_set_1.parquet"
# VAL_PATH   = "/kaggle/input/datasets/vortex07/kaggle/task_c_validation_set.parquet"
# TEST_PATH  = "/kaggle/input/datasets/vortex07/kaggle/task_c_test_set_sample.parquet"

# --- OPTION B: Download from HuggingFace (default) ---
from datasets import load_dataset
print("Downloading SemEval-2026-Task13 Subtask C from HuggingFace...")
hf_dataset = load_dataset("DaniilOr/SemEval-2026-Task13", "C")
hf_dataset['train'].to_parquet('task_c_train.parquet')
hf_dataset['validation'].to_parquet('task_c_val.parquet')
if 'test' in hf_dataset:
    hf_dataset['test'].to_parquet('task_c_test.parquet')
    TEST_PATH = 'task_c_test.parquet'
else:
    TEST_PATH = None
    print("No test split available on HuggingFace.")
TRAIN_PATH = 'task_c_train.parquet'
VAL_PATH   = 'task_c_val.parquet'
print("Done!")

print(f"Train: {TRAIN_PATH}")
print(f"Val:   {VAL_PATH}")
print(f"Test:  {TEST_PATH}")

In [ ]:
# ─── Backbone selection ──────────────────────────────────────────────
# Choose ONE backbone to run. Comment/uncomment as needed:
# BACKBONE = "microsoft/codebert-base"
# BACKBONE = "microsoft/graphcodebert-base"
BACKBONE = "microsoft/unixcoder-base"

# ─── Head architecture ───────────────────────────────────────────────
# Options: "linear" | "bilstm" | "dense_block"
HEAD_TYPE = "bilstm"

# ─── Training hyperparameters ────────────────────────────────────────
MAX_LENGTH           = 256      # 256 is 4x faster than 512; captures most patterns
BATCH_SIZE           = 16       # larger batch with shorter seqlen
GRAD_ACCUM_STEPS     = 2        # effective batch = 32
LEARNING_RATE        = 3e-5
NUM_EPOCHS           = 5        # full epochs, early stopping will cut
WARMUP_RATIO         = 0.06     # auto-scales with dataset size
WEIGHT_DECAY         = 0.01
LABEL_SMOOTHING      = 0.05     # mild smoothing; set 0.0 to disable
FP16                 = True     # T4 supports fp16 natively
GRADIENT_CHECKPOINTING = True   # needed for 512 tokens + T4 VRAM

# ─── Loss configuration ──────────────────────────────────────────────
FOCAL_GAMMA          = 2.0      # focal loss focusing parameter
MAX_CLASS_WEIGHT     = 10.0     # clip class weights
SUPCON_WEIGHT        = 0.15     # weight for contrastive loss branch
SUPCON_TEMP          = 0.07     # SupCon temperature
PROJ_DIM             = 128      # projection head dimension

# ─── LLRD ────────────────────────────────────────────────────────────
LLRD_DECAY           = 0.9      # learning-rate decay per layer

# ─── Bi-LSTM head config ─────────────────────────────────────────────
LSTM_HIDDEN           = 256     # full capacity
LSTM_LAYERS           = 2       # full capacity
LSTM_DROPOUT          = 0.3

# ─── Dense-block head config ─────────────────────────────────────────
DENSE_BLOCKS          = 3       # full capacity
DENSE_HIDDEN          = 512     # full capacity
DENSE_DROPOUT         = 0.2

# ─── Data paths are set in the cell above (HuggingFace download) ───

# ─── Data sampling ──────────────────────────────────────────
# 30% train + 15% val: good balance of quality vs speed on T4
# Set both to 1.0 for final submission run (will take 8-10 hrs)
TRAIN_SAMPLE_FRAC    = 0.30     # ~270K training samples
VAL_SAMPLE_FRAC      = 0.15     # ~30K val samples for fast eval

# ─── Misc ────────────────────────────────────────────────────────────
OUTPUT_DIR            = "taskC-improved-model"
EARLY_STOPPING_PATIENCE = 2     # stop sooner if plateau
NUM_LABELS            = 4
LABEL_NAMES           = ["human", "machine", "hybrid", "adversarial"]

print("=" * 60)
print(f"Backbone       : {BACKBONE}")
print(f"Head           : {HEAD_TYPE}")
print(f"Max length     : {MAX_LENGTH}")
print(f"Eff. batch     : {BATCH_SIZE * GRAD_ACCUM_STEPS}")
print(f"Data sampling  : train={TRAIN_SAMPLE_FRAC*100:.0f}%  val={VAL_SAMPLE_FRAC*100:.0f}%")
print(f"Focal gamma    : {FOCAL_GAMMA}")
print(f"Label smoothing: {LABEL_SMOOTHING}")
print(f"SupCon weight  : {SUPCON_WEIGHT}")
print(f"LLRD decay     : {LLRD_DECAY}")
print("=" * 60)

## 1. Data Loading & Class-Weight Computation

In [ ]:
def load_data(train_path, val_path, train_frac=1.0, val_frac=1.0):
    train_df = pd.read_parquet(train_path)
    val_df   = pd.read_parquet(val_path)

    train_df = train_df.dropna(subset=["code", "label"])
    val_df   = val_df.dropna(subset=["code", "label"])
    train_df["label"] = train_df["label"].astype(int)
    val_df["label"]   = val_df["label"].astype(int)

    if train_frac < 1.0:
        train_df = train_df.sample(frac=train_frac, random_state=42).reset_index(drop=True)
    if val_frac < 1.0:
        val_df = val_df.sample(frac=val_frac, random_state=42).reset_index(drop=True)

    print(f"Train: {len(train_df):,}   Val: {len(val_df):,}")
    print(f"Label distribution (train):\n{train_df['label'].value_counts().sort_index()}")
    print(f"\nLabel distribution (val):\n{val_df['label'].value_counts().sort_index()}")
    return train_df, val_df


def compute_balanced_weights(labels, num_labels, max_weight=10.0):
    """Compute sklearn balanced class weights, clipped."""
    weights = compute_class_weight(
        "balanced", classes=np.arange(num_labels), y=labels
    )
    weights = np.clip(weights, 1.0, max_weight)
    print(f"\nClass weights (clipped@{max_weight}):")
    for i, w in enumerate(weights):
        print(f"  {LABEL_NAMES[i]:>12s} (label {i}): {w:.4f}")
    return torch.tensor(weights, dtype=torch.float32).to(DEVICE)


train_df, val_df = load_data(TRAIN_PATH, VAL_PATH, TRAIN_SAMPLE_FRAC, VAL_SAMPLE_FRAC)
class_weights = compute_balanced_weights(train_df["label"].values, NUM_LABELS, MAX_CLASS_WEIGHT)

## 2. Loss Functions — Focal Loss + Supervised Contrastive Loss

In [ ]:
class FocalLoss(nn.Module):
    """
    Focal Loss with per-class alpha and focusing parameter gamma.
    Down-weights well-classified examples so the model concentrates
    gradient on hard minority samples (hybrid & adversarial).
    """
    def __init__(self, alpha=None, gamma=2.0, label_smoothing=0.0):
        super().__init__()
        self.gamma = gamma
        self.label_smoothing = label_smoothing
        if alpha is not None:
            self.register_buffer("alpha", alpha)
        else:
            self.alpha = None

    def forward(self, logits, targets):
        ce = F.cross_entropy(
            logits, targets, reduction="none",
            label_smoothing=self.label_smoothing,
        )
        pt = torch.exp(-ce)
        focal_weight = (1.0 - pt) ** self.gamma
        if self.alpha is not None:
            alpha_t = self.alpha[targets]
            focal_weight = alpha_t * focal_weight
        return (focal_weight * ce).mean()


class SupConLoss(nn.Module):
    """
    Supervised Contrastive Loss (Khosla et al. 2020).
    Pulls embeddings of the same class closer and pushes
    different-class embeddings apart.
    """
    def __init__(self, temperature=0.07):
        super().__init__()
        self.temperature = temperature

    def forward(self, features, labels):
        features = F.normalize(features, dim=1)
        batch_size = features.shape[0]

        labels = labels.contiguous().view(-1, 1)
        mask = torch.eq(labels, labels.T).float().to(features.device)

        similarity = torch.matmul(features, features.T) / self.temperature
        # mask out self-similarity
        logits_mask = torch.ones_like(mask) - torch.eye(batch_size, device=features.device)
        mask = mask * logits_mask

        # numerical stability
        logits_max, _ = similarity.max(dim=1, keepdim=True)
        logits = similarity - logits_max.detach()

        exp_logits = torch.exp(logits) * logits_mask
        log_prob = logits - torch.log(exp_logits.sum(1, keepdim=True) + 1e-12)

        # mean log-likelihood over positives
        mask_pos_pairs = mask.sum(1)
        mask_pos_pairs = torch.clamp(mask_pos_pairs, min=1)
        mean_log_prob = (mask * log_prob).sum(1) / mask_pos_pairs

        loss = -mean_log_prob.mean()
        return loss


print("FocalLoss & SupConLoss defined")

## 3. Custom Model — Backbone + Interchangeable Heads

In [ ]:
class BiLSTMHead(nn.Module):
    """
    Bi-LSTM classification head.
    Takes the full sequence of hidden states from the backbone,
    runs a bi-directional LSTM, and classifies from the final hidden.
    """
    def __init__(self, hidden_size, num_labels, lstm_hidden=256,
                 lstm_layers=2, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=hidden_size,
            hidden_size=lstm_hidden,
            num_layers=lstm_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if lstm_layers > 1 else 0.0,
        )
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(lstm_hidden * 2, num_labels)

    def forward(self, sequence_output):
        lstm_out, (h_n, _) = self.lstm(sequence_output)
        # concat forward & backward final hidden states
        h_fwd = h_n[-2]  # last forward layer
        h_bwd = h_n[-1]  # last backward layer
        h_cat = torch.cat([h_fwd, h_bwd], dim=1)
        h_cat = self.dropout(h_cat)
        return self.classifier(h_cat)


class DenseBlockHead(nn.Module):
    """
    Multi-Dense-Block classification head with residual connections.
    Creates a deeper, more expressive decision boundary.
    """
    def __init__(self, hidden_size, num_labels, num_blocks=3,
                 block_hidden=512, dropout=0.2):
        super().__init__()
        self.input_proj = nn.Linear(hidden_size, block_hidden)
        self.blocks = nn.ModuleList()
        for _ in range(num_blocks):
            self.blocks.append(nn.Sequential(
                nn.Linear(block_hidden, block_hidden),
                nn.LayerNorm(block_hidden),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.Linear(block_hidden, block_hidden),
                nn.LayerNorm(block_hidden),
                nn.GELU(),
                nn.Dropout(dropout),
            ))
        self.classifier = nn.Linear(block_hidden, num_labels)

    def forward(self, cls_embedding):
        x = self.input_proj(cls_embedding)
        for block in self.blocks:
            x = x + block(x)  # residual
        return self.classifier(x)


class CodeClassifierWithSupCon(nn.Module):
    """
    Backbone (CodeBERT / GraphCodeBERT / UniXcoder) +
    interchangeable classification head +
    projection head for supervised contrastive loss.
    """
    def __init__(self, backbone_name, num_labels, head_type="bilstm",
                 proj_dim=128, **head_kwargs):
        super().__init__()
        self.config = AutoConfig.from_pretrained(backbone_name)
        self.backbone = AutoModel.from_pretrained(backbone_name, config=self.config)
        hidden_size = self.config.hidden_size
        self.head_type = head_type

        # Classification head
        if head_type == "bilstm":
            self.head = BiLSTMHead(hidden_size, num_labels, **head_kwargs)
        elif head_type == "dense_block":
            self.head = DenseBlockHead(hidden_size, num_labels, **head_kwargs)
        else:
            # simple linear head (like default RobertaForSequenceClassification)
            self.head = nn.Sequential(
                nn.Dropout(0.1),
                nn.Linear(hidden_size, hidden_size),
                nn.Tanh(),
                nn.Dropout(0.1),
                nn.Linear(hidden_size, num_labels),
            )

        # Projection head for contrastive loss
        self.projector = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, proj_dim),
        )

        self.num_labels = num_labels

    def gradient_checkpointing_enable(self, gradient_checkpointing_kwargs=None):
        """Delegate gradient checkpointing to the backbone (required by HF Trainer)."""
        self.backbone.gradient_checkpointing_enable(
            gradient_checkpointing_kwargs=gradient_checkpointing_kwargs
        )

    def gradient_checkpointing_disable(self):
        """Delegate gradient checkpointing disable to the backbone."""
        self.backbone.gradient_checkpointing_disable()

    def forward(self, input_ids=None, attention_mask=None, labels=None, **kwargs):
        outputs = self.backbone(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True,
        )
        sequence_output = outputs.last_hidden_state  # [B, seqlen, H]
        cls_embedding = sequence_output[:, 0, :]      # [B, H]

        # Classification
        if self.head_type == "bilstm":
            logits = self.head(sequence_output)
        else:
            logits = self.head(cls_embedding)

        # Projection for SupCon
        proj = self.projector(cls_embedding)

        return (logits, proj)


print(f"Model components defined — head type: {HEAD_TYPE}")

## 4. Layer-wise Learning Rate Decay (LLRD)

In [ ]:
def get_llrd_optimizer(model, base_lr=3e-5, decay=0.9, weight_decay=0.01):
    """
    Build an AdamW optimizer where each transformer layer has a
    geometrically decaying learning rate. Lower layers (closer to
    embeddings) get smaller LRs; the classification / projection heads
    get the full base_lr.
    """
    opt_params = []
    no_decay = {"bias", "LayerNorm.weight", "LayerNorm.bias"}

    # 1. Backbone embeddings — lowest LR
    backbone = model.backbone
    num_layers = backbone.config.num_hidden_layers

    # Embeddings
    emb_lr = base_lr * (decay ** (num_layers + 1))
    emb_params_decay = []
    emb_params_no_decay = []
    for n, p in backbone.embeddings.named_parameters():
        if any(nd in n for nd in no_decay):
            emb_params_no_decay.append(p)
        else:
            emb_params_decay.append(p)
    opt_params.append({"params": emb_params_decay, "lr": emb_lr, "weight_decay": weight_decay})
    opt_params.append({"params": emb_params_no_decay, "lr": emb_lr, "weight_decay": 0.0})

    # 2. Each encoder layer
    for layer_idx in range(num_layers):
        layer_lr = base_lr * (decay ** (num_layers - layer_idx))
        layer = backbone.encoder.layer[layer_idx]
        decay_params = []
        no_decay_params = []
        for n, p in layer.named_parameters():
            if any(nd in n for nd in no_decay):
                no_decay_params.append(p)
            else:
                decay_params.append(p)
        opt_params.append({"params": decay_params, "lr": layer_lr, "weight_decay": weight_decay})
        opt_params.append({"params": no_decay_params, "lr": layer_lr, "weight_decay": 0.0})

    # 3. Classification head + projection head — full LR
    head_params_decay = []
    head_params_no_decay = []
    for module in [model.head, model.projector]:
        for n, p in module.named_parameters():
            if any(nd in n for nd in no_decay):
                head_params_no_decay.append(p)
            else:
                head_params_decay.append(p)
    opt_params.append({"params": head_params_decay, "lr": base_lr, "weight_decay": weight_decay})
    opt_params.append({"params": head_params_no_decay, "lr": base_lr, "weight_decay": 0.0})

    optimizer = torch.optim.AdamW(opt_params)
    print(f"LLRD optimizer: {len(opt_params)} param groups, "
          f"LR range [{emb_lr:.2e} -> {base_lr:.2e}]")
    return optimizer


print("LLRD optimizer builder defined")

## 5. Improved Trainer — Focal + SupCon + LLRD

In [ ]:
class ImprovedTrainer(Trainer):
    """
    Custom HuggingFace Trainer that:
    - Uses Focal Loss (with class weights + label smoothing)
    - Adds SupCon auxiliary loss on the projection head
    - Uses Layer-wise LR Decay optimizer
    """
    def __init__(self, *args, focal_loss_fn=None, supcon_loss_fn=None,
                 supcon_weight=0.15, **kwargs):
        super().__init__(*args, **kwargs)
        self.focal_loss_fn = focal_loss_fn
        self.supcon_loss_fn = supcon_loss_fn
        self.supcon_weight = supcon_weight

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        logits, proj = model(**inputs)

        # Focal loss
        loss_cls = self.focal_loss_fn(logits, labels)

        # SupCon loss
        loss_con = self.supcon_loss_fn(proj, labels)

        loss = (1.0 - self.supcon_weight) * loss_cls + self.supcon_weight * loss_con

        return (loss, (logits, proj)) if return_outputs else loss

    def create_optimizer(self):
        self.optimizer = get_llrd_optimizer(
            self.model,
            base_lr=self.args.learning_rate,
            decay=LLRD_DECAY,
            weight_decay=self.args.weight_decay,
        )
        return self.optimizer


def _preprocess_logits_for_metrics(logits, labels):
    """Strip projection embeddings — only pass classification logits to metrics."""
    if isinstance(logits, tuple):
        return logits[0]
    return logits


def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    preds = np.argmax(predictions, axis=1)

    accuracy = accuracy_score(labels, preds)
    # use macro F1 as the target metric for Task C
    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
        labels, preds, average="macro"
    )
    precision_w, recall_w, f1_w, _ = precision_recall_fscore_support(
        labels, preds, average="weighted"
    )

    return {
        "accuracy": accuracy,
        "macro_f1": f1_macro,
        "macro_precision": precision_macro,
        "macro_recall": recall_macro,
        "weighted_f1": f1_w,
        "weighted_precision": precision_w,
        "weighted_recall": recall_w,
    }


print("ImprovedTrainer defined")

## 6. Tokenisation & Dataset Preparation

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BACKBONE)

def tokenize_fn(examples):
    return tokenizer(
        examples["code"],
        truncation=True,
        padding=True,
        max_length=MAX_LENGTH,
        return_tensors="pt",
    )

train_dataset = Dataset.from_pandas(train_df[["code", "label"]])
val_dataset   = Dataset.from_pandas(val_df[["code", "label"]])

train_dataset = train_dataset.map(tokenize_fn, batched=True, remove_columns=["code"])
val_dataset   = val_dataset.map(tokenize_fn, batched=True, remove_columns=["code"])

train_dataset = train_dataset.rename_column("label", "labels")
val_dataset   = val_dataset.rename_column("label", "labels")

print(f"Tokenised — Train: {len(train_dataset):,}  Val: {len(val_dataset):,}")

## 7. Instantiate Model + Loss Functions

In [ ]:
# Build head kwargs
if HEAD_TYPE == "bilstm":
    head_kwargs = dict(
        lstm_hidden=LSTM_HIDDEN,
        lstm_layers=LSTM_LAYERS,
        dropout=LSTM_DROPOUT,
    )
elif HEAD_TYPE == "dense_block":
    head_kwargs = dict(
        num_blocks=DENSE_BLOCKS,
        block_hidden=DENSE_HIDDEN,
        dropout=DENSE_DROPOUT,
    )
else:
    head_kwargs = {}

model = CodeClassifierWithSupCon(
    backbone_name=BACKBONE,
    num_labels=NUM_LABELS,
    head_type=HEAD_TYPE,
    proj_dim=PROJ_DIM,
    **head_kwargs,
).to(DEVICE)

# Loss functions
focal_loss = FocalLoss(
    alpha=class_weights,
    gamma=FOCAL_GAMMA,
    label_smoothing=LABEL_SMOOTHING,
)
supcon_loss = SupConLoss(temperature=SUPCON_TEMP)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nModel: {BACKBONE} + {HEAD_TYPE} head")
print(f"Total params:     {total_params:,}")
print(f"Trainable params: {trainable_params:,}")
print(model)

## 8. Training

In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 4,  # large eval batch (no grads needed)
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    warmup_ratio=WARMUP_RATIO,                 # auto-scales with data size
    weight_decay=WEIGHT_DECAY,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",               # cosine > linear for longer runs
    fp16=FP16,
    logging_dir="./logs",
    logging_steps=100,
    evaluation_strategy="epoch",              # eval once per epoch (was every 100 steps!)
    save_strategy="epoch",                    # save once per epoch
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",         # target metric
    greater_is_better=True,
    remove_unused_columns=False,
    save_total_limit=2,
    report_to=[],
    dataloader_num_workers=4,                  # better GPU utilization
    gradient_checkpointing=GRADIENT_CHECKPOINTING,
    dataloader_pin_memory=True,
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = ImprovedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    focal_loss_fn=focal_loss,
    supcon_loss_fn=supcon_loss,
    supcon_weight=SUPCON_WEIGHT,
    preprocess_logits_for_metrics=_preprocess_logits_for_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)],
)

print("Starting training...")
trainer.train()

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"\nModel saved to {OUTPUT_DIR}")

## 9. Evaluation — Detailed Per-Class Metrics

In [ ]:
predictions = trainer.predict(val_dataset)
y_pred = np.argmax(predictions.predictions, axis=1)
y_true = predictions.label_ids

print("=" * 60)
print("CLASSIFICATION REPORT (uncalibrated argmax)")
print("=" * 60)
print(classification_report(
    y_true, y_pred, target_names=LABEL_NAMES, digits=4
))

print("\nConfusion Matrix:")
cm = confusion_matrix(y_true, y_pred)
cm_df = pd.DataFrame(cm, index=LABEL_NAMES, columns=LABEL_NAMES)
print(cm_df)

## 10. Per-Class Threshold Calibration

In [ ]:
def calibrate_thresholds(logits, true_labels, num_labels, label_names=None):
    """
    Sweep per-class probability thresholds on the validation set
    to maximise per-class F1, then compute the resulting macro-F1.

    Returns:
        best_thresholds: np.array of shape [num_labels]
    """
    probs = torch.softmax(torch.tensor(logits, dtype=torch.float32), dim=1).numpy()
    best_thresholds = np.full(num_labels, 0.5)
    label_names = label_names or [str(i) for i in range(num_labels)]

    print("\nPer-class threshold sweep:")
    for cls in range(num_labels):
        cls_true = (true_labels == cls).astype(int)
        best_f1 = 0.0
        best_t = 0.5
        for t in np.arange(0.05, 0.96, 0.02):
            cls_pred = (probs[:, cls] >= t).astype(int)
            if cls_pred.sum() == 0:
                continue
            _, _, f1, _ = precision_recall_fscore_support(
                cls_true, cls_pred, average="binary", zero_division=0
            )
            if f1 > best_f1:
                best_f1 = f1
                best_t = t
        best_thresholds[cls] = best_t
        print(f"  {label_names[cls]:>12s}: threshold={best_t:.2f}  binary-F1={best_f1:.4f}")

    return best_thresholds


def predict_with_thresholds(logits, thresholds):
    """Apply calibrated per-class thresholds."""
    probs = torch.softmax(torch.tensor(logits, dtype=torch.float32), dim=1).numpy()
    # Scale each class probability by inverse threshold, then argmax
    adjusted = probs / thresholds[np.newaxis, :]
    return np.argmax(adjusted, axis=1)


# Run calibration
best_thresholds = calibrate_thresholds(
    predictions.predictions, y_true, NUM_LABELS, LABEL_NAMES
)

y_pred_calibrated = predict_with_thresholds(predictions.predictions, best_thresholds)

print("\n" + "=" * 60)
print("CLASSIFICATION REPORT (calibrated thresholds)")
print("=" * 60)
print(classification_report(
    y_true, y_pred_calibrated, target_names=LABEL_NAMES, digits=4
))

# Compare
_, _, f1_uncal, _ = precision_recall_fscore_support(y_true, y_pred, average="macro")
_, _, f1_cal, _ = precision_recall_fscore_support(y_true, y_pred_calibrated, average="macro")
print(f"\nMacro-F1 — uncalibrated: {f1_uncal:.4f}  |  calibrated: {f1_cal:.4f}  |  delta = {f1_cal - f1_uncal:+.4f}")

print("\nCalibrated Confusion Matrix:")
cm_cal = confusion_matrix(y_true, y_pred_calibrated)
cm_cal_df = pd.DataFrame(cm_cal, index=LABEL_NAMES, columns=LABEL_NAMES)
print(cm_cal_df)

## 11. Inference on Test Set

In [ ]:
from itertools import chain
from tqdm import tqdm
from datasets import load_dataset as hf_load_dataset


@torch.no_grad()
def predict_test(model, tokenizer, parquet_path, output_path,
                 thresholds=None, max_length=512, batch_size=16, device=None):
    """
    Streaming inference on the test parquet to generate submission CSV.
    Optionally applies calibrated per-class thresholds.
    """
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    model.to(device)
    model.eval()

    ds = hf_load_dataset("parquet", data_files=parquet_path, split="train", streaming=True)
    it = iter(ds)
    first = next(it)
    if not {"ID", "code"}.issubset(first.keys()):
        raise ValueError("Test parquet must contain 'ID' and 'code' columns")
    stream = chain([first], it)

    def batcher(iterator, bs):
        buf = []
        for ex in iterator:
            buf.append(ex)
            if len(buf) == bs:
                yield buf
                buf = []
        if buf:
            yield buf

    with open(output_path, "w") as f:
        f.write("ID,prediction\n")

        for batch in tqdm(batcher(stream, batch_size), desc="Predicting"):
            codes = [row["code"] for row in batch]
            ids   = [row["ID"] for row in batch]

            enc = tokenizer(
                codes, truncation=True, padding=True,
                max_length=max_length, return_tensors="pt",
            )
            input_ids = enc["input_ids"].to(device)
            attention_mask = enc["attention_mask"].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs[0] if isinstance(outputs, tuple) else outputs

            if thresholds is not None:
                probs = torch.softmax(logits, dim=1).cpu().numpy()
                adjusted = probs / thresholds[np.newaxis, :]
                pred_labels = np.argmax(adjusted, axis=1).tolist()
            else:
                pred_labels = logits.argmax(dim=-1).cpu().tolist()

            for ex_id, pred in zip(ids, pred_labels):
                f.write(f"{ex_id},{pred}\n")

    print(f"Predictions saved to {output_path}")


# Run inference
OUT_CSV = "/kaggle/working/submission_taskC.csv"

if TEST_PATH is None:
    print("No test set available — skipping inference.")
else:

    predict_test(
        model=model,
        tokenizer=tokenizer,
        parquet_path=TEST_PATH,
        output_path=OUT_CSV,
        thresholds=best_thresholds,
        max_length=MAX_LENGTH,
        batch_size=32,
        device=DEVICE,
    )

    # Quick sanity check
    sub_df = pd.read_csv(OUT_CSV)
    print(f"\nSubmission shape: {sub_df.shape}")
    print(f"Prediction distribution:\n{sub_df['prediction'].value_counts().sort_index()}")

## 12. Experiment Summary & Next Steps

### What was done
- **Backbone**: UniXcoder — pre-trained on code with AST/DFG awareness & unified cross-modal representations
- **Head**: Bi-LSTM — captures sequential patterns across the full hidden sequence, richer than a single linear layer
- **Focal Loss (gamma=2)** with sklearn balanced class weights — forces gradient on hard hybrid/adversarial samples
- **SupCon auxiliary loss (weight=0.15)** — contrastive signal separates embeddings per class
- **LLRD (decay=0.9)** — lower layers kept stable, upper layers + heads adapted aggressively
- **Threshold calibration** — per-class probability cutoffs tuned on validation set to maximise macro-F1
- **10% data sample + 256 tokens (fast mode)** — fast iteration for prototyping; scale up for final submission

### How to run backbone ablations
Simply change `BACKBONE` in the Config cell:
```python
BACKBONE = "microsoft/codebert-base"        # baseline
BACKBONE = "microsoft/graphcodebert-base"    # +DFG awareness
BACKBONE = "microsoft/unixcoder-base"        # +unified cross-modal
```

### How to run head ablations
Change `HEAD_TYPE`:
```python
HEAD_TYPE = "linear"       # simple dense head
HEAD_TYPE = "bilstm"       # Bi-LSTM over full hidden sequence
HEAD_TYPE = "dense_block"  # residual dense blocks on [CLS]
```

### Tuning levers for hybrid/adversarial
- Increase `FOCAL_GAMMA` (try 3.0) to further suppress easy Human examples
- Increase `MAX_CLASS_WEIGHT` (try 15.0) for stronger pull on classes 2 & 3
- Increase `SUPCON_WEIGHT` (try 0.25) for stronger embedding separation
- Set `LABEL_SMOOTHING = 0.0` if minority recall drops